# Crime Incidents redux

We are going to again look at the crime data from the lesson five assignment. Ideally I would find data on the internet and have you import that, but I'm finding that open internet, permissionless resources for very large datasets are unreliable, so we're going to re-use the data from class 5 assignment and pretent that it is a very large dataset that we have to process lazily.

## Read the 'Crime_Incidents...' data into a single (lazy) dataframe then find the number of rows/records

Hint: For reading in you might want to use string substitution. The f-string `f"{some_number:02d}"` will zero pad a number so that the number 1, for instance, prints as '01'. Or alternatively you could use the `glob` package.

### Dask

In [35]:
import dask.dataframe as dd

In [36]:
from glob import glob
ddfs = []
files = glob("../Class_05/Assignment/Data/*.csv")

dtype={'CENSUS_TRACT': 'float64',
       'DISTRICT': 'float64',
       'PSA': 'float64',
       'XBLOCK': 'float64',
       'YBLOCK': 'float64',
       'WARD':'float64',
       'BID':'object',
      }

for i in files:
    tmp = dd.read_csv(i, dtype=dtype)
    ddfs.append(tmp)

In [37]:
ddf = dd.concat(ddfs)

In [38]:
ddf.columns

Index(['X', 'Y', 'CCN', 'REPORT_DAT', 'SHIFT', 'METHOD', 'OFFENSE', 'BLOCK',
       'XBLOCK', 'YBLOCK', 'WARD', 'ANC', 'DISTRICT', 'PSA',
       'NEIGHBORHOOD_CLUSTER', 'BLOCK_GROUP', 'CENSUS_TRACT',
       'VOTING_PRECINCT', 'LATITUDE', 'LONGITUDE', 'BID', 'START_DATE',
       'END_DATE', 'OBJECTID', 'OCTO_RECORD_ID'],
      dtype='object')

In [39]:
ddf['X'].count().compute()

np.int64(562317)

### Polars

In [40]:
import polars as pl

In [41]:
dft = pl.scan_csv(files[0])

In [42]:
dtypes = dft.head(3).collect().dtypes
names = dft.collect_schema().names()

In [43]:
dtp = dict(zip(names,dtypes))

In [44]:

dfs = []
for i in files:
    tmp = pl.scan_csv(i, schema=dtp)
    dfs.append(tmp)

In [45]:
df = pl.concat(dfs)

In [46]:
df.select('X').count().collect()

X
u32
562317


## Of the 10 months we are tracking, which month has the largest median trip distance?

### DASK

In [48]:
def dist(row):
    math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    pass

ddf['distance'] = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

X                               float64
Y                               float64
CCN                               int64
REPORT_DAT              string[pyarrow]
SHIFT                   string[pyarrow]
METHOD                  string[pyarrow]
OFFENSE                 string[pyarrow]
BLOCK                   string[pyarrow]
XBLOCK                          float64
YBLOCK                          float64
WARD                            float64
ANC                     string[pyarrow]
DISTRICT                        float64
PSA                             float64
NEIGHBORHOOD_CLUSTER    string[pyarrow]
BLOCK_GROUP             string[pyarrow]
CENSUS_TRACT                    float64
VOTING_PRECINCT         string[pyarrow]
LATITUDE                        float64
LONGITUDE                       float64
BID                     string[pyarrow]
START_DATE              string[pyarrow]
END_DATE                string[pyarrow]
OBJECTID                          int64
OCTO_RECORD_ID                  float64


In [47]:
ddf['month'] = ddf.tpep_pickup_datetime.dt.month

AttributeError: 'DataFrame' object has no attribute 'tpep_pickup_datetime'

In [10]:
ddf.groupby('month').trip_distance.median().compute()

ClientResponseError: 403, message='Forbidden', url='https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-06.parquet'

### Polars

In [14]:
df.group_by_dynamic(pl.col('tpep_pickup'), every="1mo").agg( 
    pl.col('trip_distance').median().alias('median_trip_distance')
).sort(by='median_trip_distance').collect()

SchemaFieldNotFoundError: tpep_pickup

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
UNION
  PLAN 0:
    Parquet SCAN [https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet]
    PROJECT */19 COLUMNS
    ESTIMATED ROWS: 2964624
  PLAN 1:
    Parquet SCAN [https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet]
    PROJECT */19 COLUMNS
    ESTIMATED ROWS: 3007526
  PLAN 2:
    Parquet SCAN [https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet]
    PROJECT */19 COLUMNS
    ESTIMATED ROWS: 3582628
  PLAN 3:
    Parquet SCAN [https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-04.parquet]
    PROJECT */19 COLUMNS
    ESTIMATED ROWS: 3514289
  PLAN 4:
    Parquet SCAN [https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-05.parquet]
    PROJECT */19 COLUMNS
    ESTIMATED ROWS: 3723833
  PLAN 5:
    Parquet SCAN [https://d37ci6vzurychx.cloudfront.net/trip-data

In [15]:
df.head().collect()

OSError: object-store error: The operation lacked the necessary privileges to complete for path : Error performing GET https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet in 97.691572ms - Server returned non-2xx status code: 403 Forbidden: <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again later, or contact the app or website owner.
<BR clear="all">
If you provide content to customers through CloudFront, you can find steps to troubleshoot and help prevent this error by reviewing the CloudFront documentation.
<BR clea

In [8]:
import dask.dataframe as dd

2. Lets read data from a public S3 bucket

In [ ]:
#-- need this for reading from S3
!pip install s3fs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 56.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2025.2.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cud

In [ ]:
ddf = dd.read_parquet(
    "s3://dask-data/nyc-taxi/nyc-2015.parquet/part.*.parquet",
    columns=["passenger_count", "tip_amount", "VendorID", "trip_distance"],
    storage_options={"anon": True},
)

3. Look at the first few rows

In [ ]:
ddf.head(3)

,passenger_count,tip_amount,VendorID,trip_distance
0,5,0.0,1,4.00
1,3,0.0,2,1.56
2,1,0.0,2,1.68


4. How many records does the data have?

In [ ]:
len(ddf)

146112989

5. Create a persisted ddf, where `tip_distance` is greater than 10

In [ ]:
long_trips = ddf.loc[ddf["trip_distance"] > 10].persist()

6. Calculate the mean, max, and min of `tip_amount` seperately and compute the result

In [ ]:
long_trips["tip_amount"].mean().compute()

6.059765222904728

In [ ]:
long_trips["tip_amount"].max().compute()

910.05

In [ ]:
long_trips["tip_amount"].min().compute()

0.0

In [9]:
ddf = dd.read_csv('https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud/data?select=creditcard.csv')

ValueError: An error occurred while calling the read_csv method registered to the pandas backend.
Original Message: Backing filesystem couldn't determine file size, cannot do chunked reads. To read, set blocksize=None.

In [10]:
import pandas as pd

df = pd.read_csv('https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud/data?select=creditcard.csv')

ParserError: Error tokenizing data. C error: Expected 1 fields in line 9, saw 2


In [11]:
import kagglehub
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

Using Colab cache for faster access to the 'creditcardfraud' dataset.


In [12]:
path

'/kaggle/input/creditcardfraud'

In [13]:
!ls

sample_data


In [15]:
import pandas as pd
# Assuming the main file is named creditcard.csv inside the downloaded directory
data_file_path = f"{path}/creditcard.csv"
df = pd.read_csv(data_file_path)
print(df.head())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [16]:
df.shape

(284807, 31)

In [17]:
path

'/kaggle/input/creditcardfraud'

In [19]:
df = pd.read_csv('sample_data/mnist_test.csv')

In [22]:
ddf = dd.read_csv('sample_data/mnist_test.csv').compute()

In [23]:
ddf.head()

,7,0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,...,0.658,0.659,0.660,0.661,0.662,0.663,0.664,0.665,0.666,0.667
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
